# 02 - Train and evaluate (honestly)

Two training runs of the **same model with the same settings**; only the split differs:

| run | trained and scored on | what the score means |
|---|---|---|
| A `original_split` | Roboflow's random split | how well it recognises scenes it has already seen |
| B `grouped_split` | group-aware split from notebook 01 | how well it works on **unseen videos** |

The gap between A and B is the amount by which the old project's score was inflated.
Run B is the model the project ships. Everything is saved to `report/02_train_eval/`.

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
# This project's repo
REPO_URL = "https://github.com/kvarun10/Smart-Restaurant-Monitoring-System.git"

import os, sys
REPO = "/content/tablewatch"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!pip -q install roboflow ultralytics
print("code version:", os.popen(f"git -C {REPO} log -1 --format=%h_%s").read().strip())

In [ ]:
# The API key comes from Colab Secrets (key icon in the left sidebar), never from the notebook itself.
# Add a secret named ROBOFLOW_API_KEY and switch on "Notebook access".
from google.colab import userdata
from roboflow import Roboflow
import yaml, json, pathlib

WORKSPACE, PROJECT, VERSION = "varun-qlsfy", "table-detection-gomjy", 10
RAW = "/content/datasets/raw"
if not os.path.isdir(f"{RAW}/train"):
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location=RAW)

from tablewatch.tracker import normalize_state
NAMES = [normalize_state(n) for n in yaml.safe_load(open(f"{RAW}/data.yaml"))["names"]]
print(NAMES)

In [ ]:
from tablewatch.data_audit import load_yolo_dataset, class_table, leakage_report, near_duplicate_report, venue_group
from tablewatch.split import group_split, write_split

# Split by RESTAURANT (venue), not by camera: valid/test hold only restaurants never seen in training.
SEED, RATIOS = 0, (0.7, 0.15, 0.15)
GROUPED = "/content/datasets/grouped"
records = load_yolo_dataset(RAW, venue_group)
assignment = group_split(records, n_classes=len(NAMES), ratios=RATIOS, seed=SEED)
GROUPED_YAML = write_split(records, assignment, GROUPED, NAMES)
grouped_records = load_yolo_dataset(GROUPED, venue_group)

# Roboflow's own data.yaml uses relative paths that break in Colab, so write a clean one.
RAW_YAML = f"{RAW}/data_clean.yaml"
open(RAW_YAML, "w").write(f"path: {RAW}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: {len(NAMES)}\nnames: {json.dumps(NAMES)}\n")
for name, recs in [("original", records), ("grouped", grouped_records)]:
    print(name, {s: v["classes"]["unoccupied_dirty"]["instances"] for s, v in class_table(recs, NAMES).items()}, "<- dirty tables per split")

In [ ]:
from ultralytics import YOLO
import torch, shutil

CFG = dict(model="yolov8s.pt", epochs=80, patience=20, imgsz=640, batch=16, seed=0)
REPORT_DIR = "/content/report/02_train_eval"
os.makedirs(REPORT_DIR, exist_ok=True)
assert torch.cuda.is_available(), "Switch the runtime to a GPU first"

def train_and_test(run_name, data_yaml):
    model = YOLO(CFG["model"])
    model.train(data=str(data_yaml), name=run_name, project="/content/runs", exist_ok=True,
                **{k: v for k, v in CFG.items() if k != "model"})
    best = YOLO(f"/content/runs/{run_name}/weights/best.pt")
    m = best.val(data=str(data_yaml), split="test", name=f"{run_name}_test", project="/content/runs", exist_ok=True)
    per_class = {m.names[c]: {"precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
                              "mAP50": float(m.box.ap50[i]), "mAP50_95": float(m.box.ap[i])}
                 for i, c in enumerate(m.box.ap_class_index)}
    result = {"run": run_name, "config": CFG, "test_mAP50": float(m.box.map50), "test_mAP50_95": float(m.box.map),
              "per_class": per_class, "classes_missing_from_test": [n for n in NAMES if n not in per_class]}
    out = f"{REPORT_DIR}/{run_name}"; os.makedirs(out, exist_ok=True)
    json.dump(result, open(f"{out}/test_metrics.json", "w"), indent=2)
    for f in ["confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
        if os.path.exists(f"{m.save_dir}/{f}"): shutil.copy(f"{m.save_dir}/{f}", out)
    for f in ["results.csv", "results.png"]:
        if os.path.exists(f"/content/runs/{run_name}/{f}"): shutil.copy(f"/content/runs/{run_name}/{f}", out)
    return result

## Run A - original (random) split

In [ ]:
res_a = train_and_test("original_split", RAW_YAML)

## Run B - group-aware split (the real result)

In [ ]:
res_b = train_and_test("grouped_split", GROUPED_YAML)

## Compare

In [ ]:
import pandas as pd
rows = []
for r in (res_a, res_b):
    row = {"run": r["run"], "mAP50 (all)": r["test_mAP50"], "mAP50-95 (all)": r["test_mAP50_95"]}
    for n in NAMES:
        pc = r["per_class"].get(n)
        row[f"{n} mAP50"] = pc["mAP50"] if pc else None
        row[f"{n} recall"] = pc["recall"] if pc else None
    rows.append(row)
cmp = pd.DataFrame(rows).set_index("run").round(3)
cmp.to_csv(f"{REPORT_DIR}/comparison.csv")
cmp.T

**Reading it:** quote run B in the README and on the resume, with the number of dirty tables in
its test split next to it. A per-class score computed on fewer than ~30 test objects is too
noisy to quote on its own; in that case say so and report the count.

In [ ]:
# Keep the shipped weights: download now (Colab deletes /content when the session ends).
from google.colab import files
shutil.copy("/content/runs/grouped_split/weights/best.pt", "/content/tablewatch_yolov8s.pt")
files.download("/content/tablewatch_yolov8s.pt")

In [ ]:
# Download the evidence, then on your PC unzip it into the repo's report/ folder and commit it.
import shutil
from google.colab import files
shutil.make_archive("/content/02_train_eval", "zip", REPORT_DIR)
files.download("/content/02_train_eval.zip")